In [2]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

# 1. Look up the target using the UniProt ID mentioned in the paper
uniprot_id = "P10415"
target_api = new_client.target
target_record = target_api.get(target_components__accession=uniprot_id)
targets = pd.DataFrame.from_dict(target_record)

# Extract the ChEMBL ID for Bcl-2
chembl_id = targets.iloc[0]['target_chembl_id']
print(f"Target ChEMBL ID for Bcl-2: {chembl_id}")

# 2. Query bioactivity data filtering for IC50 assays on this target
activity_api = new_client.activity
raw_activities = activity_api.filter(
    target_chembl_id=chembl_id, 
    standard_type="IC50"
).only([
    'molecule_chembl_id', 
    'canonical_smiles', 
    'standard_value', 
    'standard_units'
])

# Convert the raw JSON API response into a Pandas DataFrame
df = pd.DataFrame.from_dict(raw_activities)
print(f"Raw retrieved records: {len(df)}")

Target ChEMBL ID for Bcl-2: CHEMBL4860
Raw retrieved records: 3106


In [5]:
df

,canonical_smiles,molecule_chembl_id,standard_units,standard_value,units,value
0,O=c1c2ccc(S(=O)(=O)c3ccc4c(=O)n5c6ccccc6nc5c5c...,CHEMBL337153,nM,14000.0,uM,14.0
1,CN1C(=O)c2ccc3c4ccc5c(=O)n6c7ccccc7nc6c6ccc(c7...,CHEMBL137528,nM,NaN,uM,NaN
2,Brc1cccc[n+]1-c1ccc2cc(-[n+]3ccccc3Br)ccc2c1,CHEMBL340270,nM,11700.0,uM,11.7
3,Cc1cc(-c2ccccc2)[nH]c2nc3ccccc3[n+]1-2,CHEMBL137543,nM,NaN,uM,NaN
4,O=C1c2ccc[n+]3c2-c2c1ccc[n+]2CC3,CHEMBL337147,nM,NaN,uM,NaN
...,...,...,...,...,...,...
3101,C[C@@H](c1ccc(Cl)cc1)N(C(=O)/C=C/CN(C)C)c1ccc(...,CHEMBL6143694,nM,30000.0,uM,30.0
3102,C[C@@H](c1ccc(Cl)cc1)N(C(=O)/C=C/CN(C)C)c1ccc(...,CHEMBL6102006,nM,30000.0,uM,30.0
3103,C=CC(=O)N(c1ccc(O[C@@H]2CC[C@@H](NC(=O)N[C@@H]...,CHEMBL6091960,nM,30000.0,uM,30.0
3104,Cc1cc2c(C(C)C)c(O)c(O)cc2c(O)c1-c1c(C)cc2c(C(C...,CHEMBL551288,nM,2800.0,uM,2.8


In [6]:
import io
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

# --- STEP 1: CLEAN MISSING VALUES & DUPLICATES ---
# Drop rows that are missing critical features
df_clean = df.dropna(subset=["canonical_smiles", "standard_value"])

# Convert values to floats
df_clean["standard_value"] = pd.to_numeric(df_clean["standard_value"])

# Ensure units are unified to nM (ChEMBL default for standard_value is usually nM)
df_clean = df_clean[df_clean["standard_units"].str.lower() == "nm"]


# --- STEP 2: DUPLICATE RECONCILIATION ---
# The same compound might be tested across multiple assays.
# Group by the molecule ID and compute the median IC50 value to handle duplicates.
df_grouped = (
    df_clean.groupby("molecule_chembl_id")
    .agg({"canonical_smiles": "first", "standard_value": "median"})
    .reset_index()
)


# --- STEP 3: THE BIOACTIVITY CUTOFF (BINARY THRESHOLD) ---
# To train a classifier, continuous IC50 values must be binned into Active (1) and Inactive (0).
# The industry standard (and typically what hits the ~1,400 mark for Bcl-2) uses a 10 uM (10,000 nM) threshold:
# - Active: IC50 <= 10,000 nM
# - Inactive: IC50 > 10,000 nM
def assign_activity(ic50_nm):
    if ic50_nm <= 10000:  # 10 micromolar
        return 1  # Active
    else:
        return 0  # Inactive


df_grouped["Activity_Label"] = df_grouped["standard_value"].apply(
    assign_activity
)

print(
    f"Unique unified compounds after concentration mapping: {len(df_grouped)}"
)
print(f"Active balance:\n{df_grouped['Activity_Label'].value_counts()}")


# --- STEP 4: REPRESENTATION FOR ML TRAINING ---
# Convert the molecules into the matching 1024-bit Morgan3 layout
def get_morgan3_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=1024)
    arr = np.zeros((0,), dtype=np.int8)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


features = []
valid_rows = []

for idx, row in df_grouped.iterrows():
    fp = get_morgan3_features(row["canonical_smiles"])
    if fp is not None:
        features.append(fp)
        valid_rows.append(row)

# Rebuild structural matrix
df_final_meta = pd.DataFrame(valid_rows)
df_features = pd.DataFrame(
    np.array(features), columns=[f"Bit_{i}" for i in range(1024)]
)

# Combine labels + bits
chembl_ml_ready = pd.concat(
    [df_final_meta.reset_index(drop=True), df_features], axis=1
)
chembl_ml_ready.to_csv("chembl_bcl2_training_set.csv", index=False)

print(f"\nFinal ML-Ready ChEMBL Matrix Shape: {chembl_ml_ready.shape}")

Unique unified compounds after concentration mapping: 2130
Active balance:
Activity_Label
1    2030
0     100
Name: count, dtype: int64


[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerator
[02:22:56] DEPRECATION WARNING: please use MorganGenerat


Final ML-Ready ChEMBL Matrix Shape: (2130, 1028)


In [9]:
chembl_ml_ready.isnull().sum()

molecule_chembl_id    0
canonical_smiles      0
standard_value        0
Activity_Label        0
Bit_0                 0
                     ..
Bit_1019              0
Bit_1020              0
Bit_1021              0
Bit_1022              0
Bit_1023              0
Length: 1028, dtype: int64

In [ ]:
# chemble shape is (2130, 1028)
# pubchem shape is (6306, 1031)